# Task 2 – Interpolation: theory-driven model comparison

This notebook implements three candidates in a complexity ladder: (1) Gaussian Brownian bridge, (2) Brownian bridge with GARCH volatility, and (3) VAR-assisted bridge. The goal is to compare assumptions, backtest on artificial gaps, and choose a method we can defend.

### 1. Gaussian Brownian bridge

Detta är er baseline.

Den antar att log-priser rör sig som en Brownian motion lokalt, men eftersom ni känner priset före och efter gapet tvingas den simulerade vägen att sluta på rätt punkt.

Styrka: enkel, tydlig och lätt att försvara.
Svaghet: antar konstant volatilitet och ignorerar samband mellan serier.

### 2. Brownian bridge + GARCH-volatilitet

Detta är en rimlig “nästa nivå”.

Den behåller Brownian bridge-idén, men använder GARCH(1,1) för att uppskatta hur volatiliteten förändras över tid. Därmed får ni en mer realistisk osäkerhet om serien har volatility clustering.

Styrka: fortfarande begriplig, men mer finansiellt realistisk.
Svaghet: kräver att ni kan motivera GARCH-antagandet och att modellen inte blir överdriven för interpolering.

### 3. VAR-assisted bridge

Detta testar om ni kan använda insikter från Task 1, alltså relationer mellan serier.

Den använder andra seriers returns för att prediktera den saknade seriens returns under gapet, och korrigerar sedan vägen så att endpointen fortfarande matchas.

Styrka: använder korrelationer/grupperingar mellan serier.
Svaghet: mer känslig för instabila samband och missing values i andra serier.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Dict, List, Tuple
from scipy.optimize import minimize
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error

The models are tested against:
* RMSE
* MAE
* 90% coverage
* genomsnittlig bredd på osäkerhetsintervall

## 1. Load and prepare data

We work with log-prices and log-returns because log-returns add over time. This makes it possible to condition the missing path on the known price before and after the gap.

In [ ]:
DATA_PATH = "../data/spiff_data-2.csv"  # adjust if needed
MISSING_MARKERS = [1000]
raw = pd.read_csv(DATA_PATH)

def prepare_prices(df: pd.DataFrame, missing_markers: List[float] = [1000]):
    df = df.copy()
    time_col = df.columns[1] if str(df.columns[0]).lower().startswith("unnamed") else df.columns[0]
    price_cols = [c for c in df.columns if c != time_col and not str(c).lower().startswith("unnamed")]
    prices = df[price_cols].replace(missing_markers, np.nan).astype(float)
    log_prices = np.log(prices)
    log_returns = log_prices.diff()
    return df[time_col], prices, log_prices, log_returns, price_cols

time, prices, log_prices, log_returns, assets = prepare_prices(raw, MISSING_MARKERS)
print(assets)
prices.head()

## 2. Identify internal gaps

Internal gaps have a known observation immediately before and after the missing block. The final 200 missing points belong to extrapolation, not interpolation.

In [ ]:
def find_internal_gaps(s: pd.Series) -> List[Tuple[int, int]]:
    is_na = s.isna().to_numpy()
    gaps = []
    i, n = 0, len(s)
    while i < n:
        if is_na[i]:
            start = i
            while i + 1 < n and is_na[i + 1]:
                i += 1
            end = i
            if start > 0 and end < n - 1 and not is_na[start - 1] and not is_na[end + 1]:
                gaps.append((start, end))
        i += 1
    return gaps

gaps_by_asset = {asset: find_internal_gaps(prices[asset]) for asset in assets}
gaps_by_asset

## 3. Model A: Gaussian Brownian bridge

Assumption: log-return increments are locally Gaussian with constant volatility. The model conditions simulated return paths so that they exactly connect the observed price before the gap to the observed price after the gap.

In [ ]:
@dataclass
class InterpolationResult:
    asset: str
    gap: Tuple[int, int]
    method: str
    point: pd.Series
    lower: pd.Series
    upper: pd.Series
    simulations: np.ndarray

def _summary_from_log_paths(log_paths: np.ndarray, index):
    price_paths = np.exp(log_paths)
    return (
        pd.Series(np.median(price_paths, axis=0), index=index),
        pd.Series(np.quantile(price_paths, 0.05, axis=0), index=index),
        pd.Series(np.quantile(price_paths, 0.95, axis=0), index=index),
    )

def gaussian_brownian_bridge(log_prices, log_returns, asset, gap, n_sims=5000, seed=42):
    rng = np.random.default_rng(seed)
    start, end = gap
    before, after = start - 1, end + 1
    m = end - start + 1
    n_steps = m + 1
    x0, xT = log_prices.loc[before, asset], log_prices.loc[after, asset]
    total_return = xT - x0

    r = log_returns[asset].copy()
    r.loc[start:after] = np.nan  # avoid using the hidden interval
    sigma = r.dropna().std(ddof=1)

    increments = rng.normal(0.0, sigma, size=(n_sims, n_steps))
    increments = increments - increments.mean(axis=1, keepdims=True) + total_return / n_steps
    log_paths = x0 + np.cumsum(increments, axis=1)[:, :m]
    idx = log_prices.index[start:end+1]
    point, lower, upper = _summary_from_log_paths(log_paths, idx)
    return InterpolationResult(asset, gap, "Gaussian Brownian bridge", point, lower, upper, log_paths)

## 4. Model B: Brownian bridge + GARCH volatility

Assumption: the bridge logic is right, but volatility is not constant. We estimate GARCH(1,1) on information before the gap and use forecast variances inside the bridge. This is useful if the series shows volatility clustering.

In [ ]:
def fit_garch11(returns: pd.Series):
    r = returns.dropna().to_numpy(dtype=float)
    if len(r) < 50:
        raise ValueError("Too few observations to fit GARCH(1,1).")
    scale = np.std(r, ddof=1)
    x = r / scale
    var0 = np.var(x)

    def unpack(theta):
        omega = np.exp(theta[0])
        a_raw, b_raw = np.exp(theta[1]), np.exp(theta[2])
        denom = 1 + a_raw + b_raw
        return omega, 0.999 * a_raw / denom, 0.999 * b_raw / denom

    def neg_ll(theta):
        omega, alpha, beta = unpack(theta)
        h = np.empty_like(x); h[0] = max(var0, 1e-8)
        for t in range(1, len(x)):
            h[t] = max(omega + alpha * x[t-1]**2 + beta * h[t-1], 1e-10)
        return 0.5 * np.sum(np.log(h) + x**2 / h)

    res = minimize(neg_ll, x0=np.array([-4.0, -2.0, 2.0]), method="Nelder-Mead", options={"maxiter": 5000})
    omega, alpha, beta = unpack(res.x)
    h = np.empty_like(x); h[0] = max(var0, 1e-8)
    for t in range(1, len(x)):
        h[t] = max(omega + alpha * x[t-1]**2 + beta * h[t-1], 1e-10)
    return omega * scale**2, alpha, beta, h[-1] * scale**2

def forecast_garch_variances(omega, alpha, beta, last_var, horizon: int):
    out = np.empty(horizon); current = last_var
    for h in range(horizon):
        current = omega + (alpha + beta) * current
        out[h] = max(current, 1e-12)
    return out

def brownian_bridge_garch(log_prices, log_returns, asset, gap, n_sims=5000, seed=42):
    rng = np.random.default_rng(seed)
    start, end = gap
    before, after = start - 1, end + 1
    m = end - start + 1
    n_steps = m + 1
    x0, xT = log_prices.loc[before, asset], log_prices.loc[after, asset]
    total_return = xT - x0

    train_returns = log_returns.loc[:before, asset].dropna()  # no leakage from after the gap
    omega, alpha, beta, last_var = fit_garch11(train_returns)
    variances = forecast_garch_variances(omega, alpha, beta, last_var, n_steps)
    raw = rng.normal(0.0, np.sqrt(variances), size=(n_sims, n_steps))

    # Conditional Gaussian bridge with unequal variances: allocate correction proportional to variance.
    adjustment = (total_return - raw.sum(axis=1))[:, None] * (variances / variances.sum())[None, :]
    increments = raw + adjustment
    log_paths = x0 + np.cumsum(increments, axis=1)[:, :m]
    idx = log_prices.index[start:end+1]
    point, lower, upper = _summary_from_log_paths(log_paths, idx)
    return InterpolationResult(asset, gap, "Brownian bridge + GARCH volatility", point, lower, upper, log_paths)

## 5. Model C: VAR-assisted bridge

Assumption: Task 1 may show that some series move together. We use other series' observed returns to predict the missing return path, and then bridge-correct the cumulative return so the endpoint matches. This is more interpretable than a full Kalman+VAR implementation but tests the same idea: cross-series information.

In [ ]:
def make_var_features(log_returns: pd.DataFrame, target: str, lags: int = 1):
    others = [c for c in log_returns.columns if c != target]
    X_parts = []
    for lag in range(lags + 1):
        shifted = log_returns[others].shift(lag)
        shifted.columns = [f"{c}_lag{lag}" for c in others]
        X_parts.append(shifted)
    return pd.concat(X_parts, axis=1), log_returns[target]

def var_assisted_bridge(log_prices, log_returns, asset, gap, lags=1, n_sims=5000, seed=42):
    rng = np.random.default_rng(seed)
    start, end = gap
    before, after = start - 1, end + 1
    m = end - start + 1
    n_steps = m + 1
    x0, xT = log_prices.loc[before, asset], log_prices.loc[after, asset]
    total_return = xT - x0

    X, y = make_var_features(log_returns, asset, lags=lags)
    pred_idx = list(range(start, after + 1))
    train_mask = y.notna() & X.notna().all(axis=1)
    train_mask.loc[start:after] = False
    X_train, y_train = X.loc[train_mask], y.loc[train_mask]
    X_pred = X.loc[pred_idx]
    if X_pred.isna().any().any():
        raise ValueError("Predictor series are missing during this gap.")

    model = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-4, 4, 25)))
    model.fit(X_train, y_train)
    mu = model.predict(X_pred)
    sigma = (y_train - model.predict(X_train)).std(ddof=1)

    raw = rng.normal(loc=mu, scale=sigma, size=(n_sims, n_steps))
    increments = raw + ((total_return - raw.sum(axis=1)) / n_steps)[:, None]
    log_paths = x0 + np.cumsum(increments, axis=1)[:, :m]
    idx = log_prices.index[start:end+1]
    point, lower, upper = _summary_from_log_paths(log_paths, idx)
    return InterpolationResult(asset, gap, "VAR-assisted bridge", point, lower, upper, log_paths)

## 6. Backtest on artificial gaps

We hide observed windows with the same length as the real gaps. This lets us compare methods on known truth and discuss expected interpolation performance without data leakage.

In [ ]:
def mask_gap_for_backtest(prices, asset, gap):
    tmp = prices.copy(); start, end = gap
    tmp.loc[start:end, asset] = np.nan
    return tmp

def valid_backtest_gaps(prices, asset, gap_len, n_windows=20, seed=42):
    rng = np.random.default_rng(seed)
    s = prices[asset]; candidates = []
    for start in range(1, len(s) - gap_len - 1):
        end = start + gap_len - 1
        if s.loc[start-1:end+1].notna().all():
            candidates.append((start, end))
    if not candidates:
        return []
    chosen = rng.choice(len(candidates), size=min(n_windows, len(candidates)), replace=False)
    return [candidates[i] for i in chosen]

def evaluate_one_method(method_func, prices, asset, gap, **kwargs):
    tmp_prices = mask_gap_for_backtest(prices, asset, gap)
    tmp_log_prices = np.log(tmp_prices)
    tmp_log_returns = tmp_log_prices.diff()
    res = method_func(tmp_log_prices, tmp_log_returns, asset, gap, **kwargs)
    true = prices.loc[gap[0]:gap[1], asset]
    return {
        "asset": asset,
        "gap": gap,
        "method": res.method,
        "rmse": np.sqrt(mean_squared_error(true, res.point)),
        "mae": mean_absolute_error(true, res.point),
        "coverage90": ((true >= res.lower) & (true <= res.upper)).mean(),
        "avg_width90": (res.upper - res.lower).mean(),
    }

def run_backtest(prices, assets, methods, n_windows=10, seed=42):
    rows = []
    for asset in assets:
        real_gaps = find_internal_gaps(prices[asset])
        if not real_gaps: continue
        gap_len = real_gaps[0][1] - real_gaps[0][0] + 1
        for gap in valid_backtest_gaps(prices, asset, gap_len, n_windows=n_windows, seed=seed):
            for name, func in methods.items():
                try:
                    rows.append(evaluate_one_method(func, prices, asset, gap, seed=seed))
                except Exception as e:
                    rows.append({"asset": asset, "gap": gap, "method": name, "error": str(e)})
    return pd.DataFrame(rows)

methods = {
    "Gaussian Brownian bridge": gaussian_brownian_bridge,
    "Brownian bridge + GARCH volatility": brownian_bridge_garch,
    "VAR-assisted bridge": var_assisted_bridge,
}
backtest_results = run_backtest(prices, assets, methods, n_windows=10, seed=123)
backtest_results

In [ ]:
summary = (
    backtest_results.dropna(subset=["rmse"])
    .groupby("method")
    .agg(rmse_mean=("rmse", "mean"), mae_mean=("mae", "mean"),
         coverage90_mean=("coverage90", "mean"), avg_width90_mean=("avg_width90", "mean"),
         n_tests=("rmse", "count"))
    .sort_values("rmse_mean")
)
summary

## 7. Interpolate real gaps and plot uncertainty

Choose the final method after reading the backtest table. A good rule: use Brownian bridge as baseline, upgrade to GARCH if volatility clustering matters and improves calibration, use VAR-assisted bridge only if cross-series relationships help in backtests.

In [ ]:
CHOSEN_METHOD = brownian_bridge_garch  # change after reviewing backtest_results
real_results = []
for asset, gaps in gaps_by_asset.items():
    for gap in gaps:
        try:
            real_results.append(CHOSEN_METHOD(log_prices, log_returns, asset, gap, seed=123))
        except Exception as e:
            print(f"{asset} {gap}: {e}")

filled_prices = prices.copy()
uncertainty_rows = []
for res in real_results:
    filled_prices.loc[res.point.index, res.asset] = res.point
    for idx in res.point.index:
        uncertainty_rows.append({"asset": res.asset, "index": idx, "method": res.method,
                                 "estimate": res.point.loc[idx], "lower_90": res.lower.loc[idx], "upper_90": res.upper.loc[idx]})
uncertainty = pd.DataFrame(uncertainty_rows)
uncertainty.head()

In [ ]:
def plot_interpolation(prices, res, window=30):
    asset = res.asset; start, end = res.gap
    left, right = max(0, start-window), min(len(prices)-1, end+window)
    plt.figure(figsize=(10, 5))
    plt.plot(prices.index[left:right+1], prices.loc[left:right, asset], label="Observed")
    plt.plot(res.point.index, res.point, label="Interpolated median")
    plt.fill_between(res.point.index, res.lower, res.upper, alpha=0.25, label="90% interval")
    plt.axvspan(start, end, alpha=0.10)
    plt.title(f"{asset}: {res.method}")
    plt.xlabel("Index"); plt.ylabel("Price"); plt.legend(); plt.show()

for res in real_results:
    plot_interpolation(prices, res)

## SUMMARY

We evaluated interpolation methods according to a complexity ladder. The Gaussian Brownian bridge was used as a transparent baseline because it directly conditions the missing path on observed prices before and after the gap. A GARCH bridge was added to account for volatility clustering while retaining the same conditional structure. Finally, a VAR-assisted bridge tested whether cross-series relationships from the data analysis improved interpolation. The final method was selected based on artificial-gap backtesting, uncertainty calibration, and interpretability, not model complexity alone.